In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np, os
import matplotlib.pyplot as plt

from pathlib import Path
import datetime
import warnings
# import yaml
# import re
# import datetime

# import pyarrow as pa
# import pyarrow.parquet as pq

# import gbd_mapping as gm
# from vivarium import Artifact

# from db_queries import get_ids, get_outputs, get_population, get_covariate_estimates
# from get_draws.api import get_draws

import vivarium_helpers as vh
# import vivarium_helpers.id_helper as idh
# from vivarium_helpers.vph_output.operations import VPHOperator
# from vivarium_helpers.vph_output.measures import VPHResults
from vivarium_helpers.utils import convert_to_categorical, constant_categorical, print_memory_usage, Timer
# from vivarium_helpers.vph_output import cleaning
# from vivarium_helpers.vph_output.loading import load_draws_from_keyspace_files
from vivarium_helpers.projects.alzheimers import loading, population
from vivarium_helpers.projects.alzheimers.population import RunType
# from vivarium_helpers.vph_artifact.operations import convert_to_sim_format
from vivarium_helpers.projects.alzheimers.results import AlzheimersResultsProcessor

!date
!whoami
!uname -a
!pwd

Thu Nov 19 18:45:01 PST 2026


lutzes


Linux long-slurm-sarchive-p0115 5.4.0-216-generic #236-Ubuntu SMP Fri Apr 11 19:53:21 UTC 2025 x86_64 x86_64 x86_64 GNU/Linux


/ihme/homes/lutzes/vivarium_research_alzheimers/results_tables


In [3]:
# from vivarium_helpers.projects.alzheimers.results import AlzheimersResultsProcessor

# Record version of Vivarium Helpers

In [4]:
!conda list | grep vivarium

# packages in environment at /ihme/homes/lutzes/.conda/envs/vivarium_csu_alzheimers_artifact:
vivarium-artifact         1.1.1                    pypi_0    pypi
vivarium-build-utils      4.11.1                   pypi_0    pypi
vivarium-cluster-tools    4.8.0                    pypi_0    pypi
vivarium-config-tree      5.2.1                    pypi_0    pypi
vivarium-csu-alzheimers   0.1.dev76+g982fe3fb1          pypi_0    pypi
vivarium-dependencies     1.3.1                    pypi_0    pypi
vivarium-engine           6.0.1                    pypi_0    pypi
vivarium-gbd-access       7.0.1                    pypi_0    pypi
vivarium-gbd-mapping      6.0.7                    pypi_0    pypi
vivarium-helpers          0.5.0.dev251111          pypi_0    pypi
vivarium-inputs           9.1.3                    pypi_0    pypi
vivarium-public-health    6.8.0                    pypi_0    pypi
vivarium-risk-distributions 3.2.1                    pypi_0    pypi


In [5]:
!pip freeze | grep vivarium

vivarium-artifact==1.1.1
vivarium-build-utils==4.11.1
vivarium-cluster-tools==4.8.0
vivarium-config-tree==5.2.1
-e git+https://github.com/ihmeuw/vivarium_csu_alzheimers.git@982fe3fb1cfe03824fcf5cf945850e1c9b0688b8#egg=vivarium_csu_alzheimers
vivarium-dependencies==1.3.1
vivarium-engine==6.0.1
vivarium-gbd-mapping==6.0.7
vivarium-public-health==6.8.0
vivarium-risk-distributions==3.2.1
vivarium_gbd_access==7.0.1
-e git+https://github.com/ihmeuw/vivarium_helpers.git@5b35fe7c0e45dd7296841aa977c3d299d1fdcaac#egg=vivarium_helpers
vivarium_inputs==9.1.3


# Define data directories

In [6]:
# Project directory
project_dir = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/')

# Output directory to store final results, in a folder named with
# today's date. These files include draws. write_final_csvs.ipynb writes
# the client files without draws to the no_draws folder next to it.
results_date = datetime.date.today().strftime('%Y_%m_%d')
output_dir = project_dir / 'results' / 'final_csvs' / results_date / 'draws'
output_dir.mkdir(parents=True, exist_ok=True)


def save_csv(df, filename):
    """Write df to output_dir, with a warning if the file already exists."""
    path = output_dir / filename
    if path.exists():
        warnings.warn(f"Overwriting existing file {path}")
    df.to_csv(path, index=False)


def drop_empty_rate_rows(df):
    """Drop the rate rows that have no measure. do_final_processing adds
    these rows for scenarios that do not have the measure (for example,
    no BBBM tests in the Reference scenario). These rows have no values.
    """
    empty = df['Measure'].isna()
    value_cols = ['Mean', '95% UI Lower', '95% UI Upper'] + [c for c in df if c.startswith('draw_')]
    assert df.loc[empty, value_cols].isna().all().all(), "Rows with no measure have values"
    return df[~empty]


# Output from multistate life table
# mslt_output_dir = output_dir.parent / 'results_12_4_2025_mslt'
mslt_output_dir = project_dir / 'results' / 'mslt_draws'
print(mslt_output_dir.exists())
output_dir

True


PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/final_csvs/2026_11_19/draws')

In [7]:
# # For testing: Run directory containing model 8.3 results for all
# # locations
# model_run_subdir = 'results/abie_consistent_model_test/united_states_of_america/2025_10_28_08_55_05/'

# # Results directory for model 8.3, for testing
# results_dirs = project_dir / model_run_subdir / 'results/'

# Artifact for models 10.0 - 12.2 was '10.0'
# Artifact for models 13.0 and 14.0
artifact_model_number = '13.0'


In [8]:
# Model 8.4 results (final runs for 10/31/25 intermediate results)
# Each batch run contains all locations, 100 random seeds, and 3 or 4 draws
model_run_dir_8_4 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model8.4/model_spec')

# Model 8.7 results (updated final run completed on 11/10/2025)
model_run_dir_8_7 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model8.7/model_spec')

# Model 12.2 results (updated final run completed on 02/14/2026)
model_run_dir_12_2 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model12.2')

# Model 13.0 results (rebuilt artifacts, run started 10/29/2026)
# This run has a single run directory. The loader adds the 'results'
# subfolder to each run directory.
# The model 13.0 run is V&V sized (5 seeds of 20,000 simulants) and has
# only 3 locations, so use the V&V run type for the model scale.
run_type_13_0 = RunType.V_AND_V
locations_13_0 = ['United States of America', 'China', 'Brazil']
model_run_dir_13_0 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model13.0/model_spec/2026_10_29_14_47_34')

# Model 14.0 results (new testing ramps and scenarios, run started 10/30/2026)
# Same artifacts as model 13.0. This run is also V&V sized (5 seeds of
# 20,000 simulants), with all 10 locations.
run_type_14_0 = RunType.V_AND_V
locations_14_0 = loading.LOCATIONS
model_run_dir_14_0 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model14.0/model_spec/2026_10_30_13_30_29')

# Model 16.0 results (production run started 11/18/2026)
# Same artifacts as model 13.0 and the same scenarios as model 14.0.
# This run is production sized (100 seeds of 20,000 simulants), with all
# 10 locations and 25 draws.
run_type_16_0 = RunType.FINAL
locations_16_0 = loading.LOCATIONS
model_run_dir_16_0 = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08')

# Batch runs are stored in "timestamp" subdirectories of the model run
# directory.
# In model 12.2, the run directories are nested under two more levels of
# subfolders: batch number, then model_spec, then timestamp.

# Select a model and filter for testing
# batch_run_dirs = loading.list_batch_run_directories(model_run_dir_12_2)#[:2]
# batch_run_dirs = [str(model_run_dir_13_0)]
# batch_run_dirs = [str(model_run_dir_14_0)]
batch_run_dirs = [str(model_run_dir_16_0)]

# batch_results_dirs
batch_run_dirs

['/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08']

# Create an AlzheimersResultsProcessor object to process results

In [9]:
results = AlzheimersResultsProcessor(
    artifact_model_number,
    run_type_16_0,
    locations=locations_16_0,
    batch_run_dirs=batch_run_dirs,
)

In [10]:
results.initial_simulation_population

2000000

In [11]:
results.locations

['United States of America',
 'Brazil',
 'China',
 'Germany',
 'Israel',
 'Japan',
 'Spain',
 'Sweden',
 'Taiwan',
 'United Kingdom']

In [12]:
results.draws

[202,
 170,
 164,
 135,
 243,
 181,
 197,
 35,
 11,
 75,
 41,
 26,
 216,
 105,
 15,
 198,
 233,
 179,
 19,
 156,
 172,
 90,
 96,
 177,
 24]

# Load population data and calculate model scale

In [13]:
with Timer():
    results.load_population_data(append_aggregates=False)
results.ops.describe(results.model_scale)

Elapsed time: 0:00:14.387068


,count,mean,std,min,25%,50%,75%,max
location,,,,,,,,
Brazil,25.0,0.666241,0.048702,0.569066,0.647484,0.664228,0.697841,0.746260
China,25.0,0.074932,0.004991,0.066343,0.071189,0.073947,0.080082,0.082505
Germany,25.0,0.564469,0.046950,0.483817,0.541823,0.565487,0.574408,0.692730
Israel,25.0,26.437376,2.594453,21.371444,25.341007,26.388861,27.918803,33.464651
Japan,25.0,0.369508,0.022974,0.323476,0.355148,0.364880,0.390183,0.409644
Spain,25.0,1.898532,0.096439,1.775293,1.819973,1.873009,1.940022,2.072102
Sweden,25.0,7.868337,0.443373,7.053982,7.624894,7.863023,8.151224,8.754205
Taiwan,25.0,5.241038,0.277182,4.550281,5.137899,5.242813,5.410650,5.660475
United Kingdom,25.0,1.270413,0.060722,1.121093,1.232772,1.271746,1.314122,1.389850


In [14]:
results.person_time

,location,sex,event_year,input_draw,value,age_group
0,United States of America,Female,2025,202,1.128934e+07,25_to_29
1,United States of America,Female,2025,170,1.033839e+07,25_to_29
2,United States of America,Female,2025,164,1.120331e+07,25_to_29
3,United States of America,Female,2025,135,1.001506e+07,25_to_29
4,United States of America,Female,2025,243,1.131052e+07,25_to_29
...,...,...,...,...,...,...
569995,United Kingdom,Male,2100,172,1.325974e+05,95_plus
569996,United Kingdom,Male,2100,90,1.267402e+05,95_plus
569997,United Kingdom,Male,2100,96,1.354910e+05,95_plus
569998,United Kingdom,Male,2100,177,1.356587e+05,95_plus


In [15]:
results.person_time.input_draw.unique()

array([202, 170, 164, 135, 243, 181, 197,  35,  11,  75,  41,  26, 216,
       105,  15, 198, 233, 179,  19, 156, 172,  90,  96, 177,  24],
      dtype=int16)

In [16]:
results.person_time.age_group.unique()

['25_to_29', '30_to_34', '35_to_39', '40_to_44', '45_to_49', ..., '75_to_79', '80_to_84', '85_to_89', '90_to_94', '95_plus']
Length: 15
Categories (15, object): ['25_to_29' < '30_to_34' < '35_to_39' < '40_to_44' ... '80_to_84' < '85_to_89' < '90_to_94' < '95_plus']

# Load prevalence and append counterfactual person-time

This calculates counterfactual person-time in treatment scenario for correct rate calculations.

In [17]:
with Timer():
    results.load_prevalence_and_append_counterfactual_person_time(append_aggregates=True)
results.person_time

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025')]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


36867.612507 MB after read_parquet


234.618534 MB after aggregating seeds and converting dtypes
234.618534 MB total
234.618534 MB after enforcing dtypes
Elapsed time: 1366.4715 seconds


223.448606 MB measure


268.136474 MB minuend
44.696474 MB subtrahend


191.529254 MB minuend re-indexed
31.929254 MB subtrahend re-indexed


... (2 outputs left out)

/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Elapsed time: 0:23:10.646270


,location,sex,event_year,input_draw,value,age_group,scenario
0,United States of America,Female,2025,202,1.063863e+07,65_to_69,baseline
1,United States of America,Female,2025,170,9.662163e+06,65_to_69,baseline
2,United States of America,Female,2025,164,1.057504e+07,65_to_69,baseline
3,United States of America,Female,2025,135,9.369922e+06,65_to_69,baseline
4,United States of America,Female,2025,243,1.067762e+07,65_to_69,baseline
...,...,...,...,...,...,...,...
6041995,United States of America,Both,2100,243,8.062193e+07,65_plus,baseline
6041996,United States of America,Both,2100,243,8.071912e+07,65_plus,intermediate_testing_with_treatment
6041997,United States of America,Both,2100,243,8.062193e+07,65_plus,intermediate_testing_zero_impact_treatment
6041998,United States of America,Both,2100,243,8.071912e+07,65_plus,slow_testing_with_treatment


# Load and process deaths data for testing

In [18]:
# with Timer():
#     results.load_measure_from_batch_runs('deaths', filters=loading.FINAL_RESULTS_FILTERS['deaths'])

In [19]:
# results.data.deaths

In [20]:
# deaths_prepped = results.process_deaths(results.data.deaths)
# deaths_prepped

# Preprocess MSLT results

In [21]:
!ls $mslt_output_dir

v4_2026_01_14_outside_sim_results_draw_level.csv
v4_2026_02_03_outside_sim_results_draw_level.csv
v4_2026_02_18_outside_sim_results_draw_level.csv
v4_2026_02_26_outside_sim_results_draw_level.csv
v5_2026_10_28_outside_sim_results_draw_level.csv
v6_2026_11_03_outside_sim_results_draw_level.csv
v7_2026_11_04_outside_sim_results_draw_level.csv
v8_2026_11_19_outside_sim_results_draw_level.csv


In [22]:
# Use the most recent MSLT v8 draw-level file
mslt_file = max(mslt_output_dir.glob('v8_*_outside_sim_results_draw_level.csv'))
print(mslt_file)
mslt_results = pd.read_csv(mslt_file, index_col=0)
mslt_results

/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/mslt_draws/v8_2026_11_19_outside_sim_results_draw_level.csv


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Draw,Value
0,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_105,0.000000
1,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_11,0.000000
2,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_135,0.000000
3,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_15,0.000000
4,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_156,0.000000
...,...,...,...,...,...,...,...,...,...,...
113995,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Improper Medication Uses,Number,draw_35,2164.945160
113996,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Improper Medication Uses,Number,draw_41,2092.434909
113997,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Improper Medication Uses,Number,draw_75,2051.974214
113998,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Improper Medication Uses,Number,draw_90,2003.006205


In [23]:
mslt_results.Age.unique()

array(['65_to_69', '70_to_74', '75_to_79'], dtype=object)

In [24]:
mslt_results.Draw.unique()

array(['draw_105', 'draw_11', 'draw_135', 'draw_15', 'draw_156',
       'draw_164', 'draw_170', 'draw_172', 'draw_177', 'draw_179',
       'draw_181', 'draw_19', 'draw_197', 'draw_198', 'draw_202',
       'draw_216', 'draw_233', 'draw_24', 'draw_243', 'draw_26',
       'draw_35', 'draw_41', 'draw_75', 'draw_90', 'draw_96'],
      dtype=object)

In [25]:
mslt_results.Scenario.unique()

array(['Slow Testing, No Treatment', 'Slow Testing, With Treatment',
       'Intermediate Testing, No Treatment',
       'Intermediate Testing, With Treatment',
       'Accelerated Testing, No Treatment',
       'Accelerated Testing, With Treatment'], dtype=object)

In [26]:
# Keep only the locations in the simulation run. MSLT v8 has the same 10
# locations and the same 25 draws as the model 16.0 production run.
mslt_results = mslt_results[mslt_results['Location'].isin(results.locations)]
mslt_results_prepped = results.process_mslt_results(mslt_results)
mslt_results_prepped

,age_group,disease_stage,event_year,input_draw,location,measure,metric,scenario,sex,value
0,65_to_69,Susceptible,2025,11,Brazil,BBBM Tests,Number,slow_testing_zero_impact_treatment,Female,0.000000
1,70_to_74,Susceptible,2025,11,Brazil,BBBM Tests,Number,slow_testing_zero_impact_treatment,Female,0.000000
2,75_to_79,Susceptible,2025,11,Brazil,BBBM Tests,Number,slow_testing_zero_impact_treatment,Female,0.000000
3,65_to_69,Susceptible,2025,11,Brazil,BBBM Tests,Number,slow_testing_zero_impact_treatment,Male,0.000000
4,70_to_74,Susceptible,2025,11,Brazil,BBBM Tests,Number,slow_testing_zero_impact_treatment,Male,0.000000
...,...,...,...,...,...,...,...,...,...,...
2051995,70_to_74,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Female,678.402772
2051996,75_to_79,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Female,656.663384
2051997,65_to_69,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Male,821.539394
2051998,70_to_74,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Male,619.059922


In [27]:
mslt_results_prepped.measure.unique()

['BBBM Tests', 'Positive BBBM Tests', 'Medication Initiation']
Categories (3, object): ['BBBM Tests', 'Medication Initiation', 'Positive BBBM Tests']

# BBBM Test Counts

In [28]:
filename = 'counts_bbbm_tests'
# short_name = 'bbbm_tests'
with Timer():
    bbbm_tests = results.load_measure_from_batch_runs(
        filename, False, filters=loading.FINAL_RESULTS_FILTERS[filename])

bbbm_tests

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025'), ('bbbm_test_results', '!=', 'not_tested'), ('age_group', 'in', ['65_to_69', '70_to_74', '75_to_79']), ('scenario', '!=', 'baseline')]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


4651.212572 MB after read_parquet


30.102603 MB after aggregating seeds and converting dtypes
30.102603 MB total
30.102603 MB after enforcing dtypes
Elapsed time: 889.1325 seconds
Elapsed time: 0:14:49.132678


,age_group,artifact_path,bbbm_test_results,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,value,location
0,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,positive,bbbm_testing,testing,2025,11,counts_bbbm_tests,slow_testing_zero_impact_treatment,Female,,0.0,Japan
1,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,positive,bbbm_testing,testing,2025,11,counts_bbbm_tests,slow_testing_zero_impact_treatment,Male,,0.0,Japan
2,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,positive,bbbm_testing,testing,2025,11,counts_bbbm_tests,intermediate_testing_zero_impact_treatment,Female,,0.0,Japan
3,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,positive,bbbm_testing,testing,2025,11,counts_bbbm_tests,intermediate_testing_zero_impact_treatment,Male,,0.0,Japan
4,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,positive,bbbm_testing,testing,2025,11,counts_bbbm_tests,accelerated_testing_zero_impact_treatment,Female,,0.0,Japan
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1367995,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,negative,bbbm_testing,testing,2100,243,counts_bbbm_tests,slow_testing_with_treatment,Male,,14504.0,Israel
1367996,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,negative,bbbm_testing,testing,2100,243,counts_bbbm_tests,intermediate_testing_with_treatment,Female,,25386.0,Israel
1367997,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,negative,bbbm_testing,testing,2100,243,counts_bbbm_tests,intermediate_testing_with_treatment,Male,,14504.0,Israel
1367998,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,negative,bbbm_testing,testing,2100,243,counts_bbbm_tests,accelerated_testing_with_treatment,Female,,25386.0,Israel


In [29]:
bbbm_tests.age_group.unique()

['65_to_69', '70_to_74', '75_to_79']
Categories (15, object): ['25_to_29' < '30_to_34' < '35_to_39' < '40_to_44' ... '80_to_84' < '85_to_89' < '90_to_94' < '95_plus']

In [30]:
bbbm_tests.query("value>0").age_group.unique()

['65_to_69', '70_to_74', '75_to_79']
Categories (15, object): ['25_to_29' < '30_to_34' < '35_to_39' < '40_to_44' ... '80_to_84' < '85_to_89' < '90_to_94' < '95_plus']

In [31]:
bbbm_tests.bbbm_test_results.unique()

['positive', 'negative']
Categories (3, object): ['not_tested', 'positive', 'negative']

In [32]:
bbbm_tests.query("value>0").scenario.unique()

['slow_testing_zero_impact_treatment', 'intermediate_testing_zero_impact_treatment', 'accelerated_testing_zero_impact_treatment', 'slow_testing_with_treatment', 'intermediate_testing_with_treatment', 'accelerated_testing_with_treatment']
Categories (7, object): ['baseline' < 'slow_testing_zero_impact_treatment' < 'intermediate_testing_zero_impact_treatment' < 'accelerated_testing_zero_impact_treatment' < 'slow_testing_with_treatment' < 'intermediate_testing_with_treatment' < 'accelerated_testing_with_treatment']

In [33]:
bbbm_tests_prepped = results.process_bbbm_tests(bbbm_tests, mslt_results_prepped)
bbbm_tests_prepped

,age_group,event_year,input_draw,location,measure,scenario,sex,value,disease_stage,metric
0,65_to_69,2025,11,Japan,BBBM Tests,slow_testing_zero_impact_treatment,Female,0.000000,Preclinical AD,Number
1,65_to_69,2025,11,Japan,BBBM Tests,slow_testing_zero_impact_treatment,Male,0.000000,Preclinical AD,Number
2,65_to_69,2025,11,Japan,BBBM Tests,intermediate_testing_zero_impact_treatment,Female,0.000000,Preclinical AD,Number
3,65_to_69,2025,11,Japan,BBBM Tests,intermediate_testing_zero_impact_treatment,Male,0.000000,Preclinical AD,Number
4,65_to_69,2025,11,Japan,BBBM Tests,accelerated_testing_zero_impact_treatment,Female,0.000000,Preclinical AD,Number
...,...,...,...,...,...,...,...,...,...,...
2735995,70_to_74,2100,243,United States of America,Positive BBBM Tests,accelerated_testing_with_treatment,Female,848.003465,Susceptible,Number
2735996,75_to_79,2100,243,United States of America,Positive BBBM Tests,accelerated_testing_with_treatment,Female,820.829230,Susceptible,Number
2735997,65_to_69,2100,243,United States of America,Positive BBBM Tests,accelerated_testing_with_treatment,Male,1026.924243,Susceptible,Number
2735998,70_to_74,2100,243,United States of America,Positive BBBM Tests,accelerated_testing_with_treatment,Male,773.824903,Susceptible,Number


In [34]:
bbbm_tests_prepped.disease_stage.unique()

['Preclinical AD', 'Susceptible']
Categories (2, object): ['Preclinical AD', 'Susceptible']

In [35]:
bbbm_tests_prepped.measure.unique()

['BBBM Tests', 'Positive BBBM Tests']
Categories (2, object): ['BBBM Tests', 'Positive BBBM Tests']

In [36]:
with Timer():
    # bbbm_tests_output = results.summarize_and_beautify(bbbm_tests_prepped, 'disease_stage')
    # process_* returns event_year as an unordered categorical, and the
    # event_year >= 2025 filter in do_final_processing cannot compare it
    # with the installed pandas version, so cast it back to int16 first
    bbbm_tests_output = results.do_final_processing(bbbm_tests_prepped.astype({'event_year': 'int16'}), include_draws=True)
    bbbm_tests_output = drop_empty_rate_rows(bbbm_tests_output)
bbbm_tests_output
# 121600 rows

2026-11-19 19:23:34.750502


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Timer SummarizingTimer started


Elapsed time: 211.2252 seconds
2026-11-19 19:27:27.151595
Elapsed time: 0:03:52.481362


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:994: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace(column_value_map)


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243
0,2025,Brazil,65_plus,Both,Preclinical AD,"Accelerated Testing, With Treatment",BBBM Tests,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2025,Brazil,65_plus,Female,Preclinical AD,"Accelerated Testing, With Treatment",BBBM Tests,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2025,Brazil,65_plus,Male,Preclinical AD,"Accelerated Testing, With Treatment",BBBM Tests,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2025,Brazil,65_plus,Both,Preclinical AD,"Accelerated Testing, No Treatment",BBBM Tests,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2025,Brazil,65_plus,Female,Preclinical AD,"Accelerated Testing, No Treatment",BBBM Tests,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444595,2100,United States of America,75_to_79,Female,Susceptible,"Slow Testing, With Treatment",Positive BBBM Tests,Rate,0.000367,0.000364,...,0.000371,0.000367,0.000369,0.000367,0.000364,0.000367,0.000367,0.000369,0.000369,0.000369
444596,2100,United States of America,75_to_79,Male,Susceptible,"Slow Testing, With Treatment",Positive BBBM Tests,Rate,0.000396,0.000393,...,0.000399,0.000395,0.000396,0.000397,0.000396,0.000399,0.000398,0.000399,0.000398,0.000394
444597,2100,United States of America,75_to_79,Both,Susceptible,"Slow Testing, No Treatment",Positive BBBM Tests,Rate,0.000380,0.000378,...,0.000384,0.000380,0.000381,0.000381,0.000379,0.000382,0.000382,0.000383,0.000382,0.000381
444598,2100,United States of America,75_to_79,Female,Susceptible,"Slow Testing, No Treatment",Positive BBBM Tests,Rate,0.000367,0.000364,...,0.000371,0.000367,0.000369,0.000367,0.000364,0.000368,0.000367,0.000369,0.000369,0.000369


In [37]:
bbbm_tests_output.Measure.unique()

['BBBM Tests', 'Positive BBBM Tests']
Categories (2, object): ['BBBM Tests', 'Positive BBBM Tests']

In [38]:
bbbm_tests_output.Age.unique()

['65_plus', '65_to_69', '70_to_74', '75_to_79']
Categories (4, object): ['65_plus', '65_to_69', '70_to_74', '75_to_79']

In [39]:
bbbm_tests_output.query("Mean>0").Scenario.unique()

['Accelerated Testing, With Treatment', 'Accelerated Testing, No Treatment', 'Intermediate Testing, With Treatment', 'Intermediate Testing, No Treatment', 'Slow Testing, With Treatment', 'Slow Testing, No Treatment']
Categories (7, object): ['Accelerated Testing, With Treatment', 'Accelerated Testing, No Treatment', 'Reference', 'Intermediate Testing, With Treatment', 'Intermediate Testing, No Treatment', 'Slow Testing, With Treatment', 'Slow Testing, No Treatment']

In [40]:
bbbm_tests_output.query(
    "Year == 2100 & Location == 'United States of America' & Age == '75_to_79' & Sex == 'Male' & `Disease Stage` == 'Susceptible' & Scenario == 'BBBM Testing Only' & Measure == 'Positive BBBM Tests'"
)

/tmp/ipykernel_872078/458690447.py:1: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  bbbm_tests_output.query(


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243


### Testing some stuff...

In [41]:
# temp = results.do_final_processing(bbbm_tests_prepped)
# temp

In [42]:
# results.ops.assert_values_equal(temp, bbbm_tests_output, value_cols=['Mean', '95% UI Lower', '95% UI Upper'], check_exact=False)

In [43]:
# temp2 = results.do_final_processing(bbbm_tests_prepped, include_draws=False)
# temp2

In [44]:
# results.ops.assert_values_equal(temp2, bbbm_tests_output, value_cols=['Mean', '95% UI Lower', '95% UI Upper'], check_exact=False)

In [45]:
# temp3 = results.do_final_processing(bbbm_tests_prepped, include_draws=True)
# temp3

In [46]:
# results.ops.assert_values_equal(
#     temp3.drop(columns=[c for c in temp3.columns if c.startswith('draw_')]),
#     bbbm_tests_output, value_cols=['Mean', '95% UI Lower', '95% UI Upper'], check_exact=False)

In [47]:
# Save BBBM testing results
save_csv(bbbm_tests_output, 'bbbm_tests.csv')

# CSF and PET Test Counts

In [48]:
# csf_pet_filters = [('testing_state', 'in', ['csf', 'pet'])]
filename = 'counts_baseline_tests_among_eligible'
with Timer():
    csf_pet_tests = results.load_measure_from_batch_runs(
        filename, False, filters=loading.FINAL_RESULTS_FILTERS[filename]
    )
print(len(csf_pet_tests), 'rows')
csf_pet_tests.head()

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025'), ('testing_state', 'in', ['csf', 'pet'])]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


25323.21271 MB after read_parquet


163.862741 MB after aggregating seeds and converting dtypes
163.862741 MB total
163.862741 MB after enforcing dtypes
Elapsed time: 854.5959 seconds
Elapsed time: 0:14:14.596106
7448000 rows


,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,testing_state,value,location
0,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2025,11,counts_baseline_tests_among_eligible,baseline,Female,,csf,0.0,Japan
1,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2025,11,counts_baseline_tests_among_eligible,baseline,Female,,pet,0.0,Japan
2,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2025,11,counts_baseline_tests_among_eligible,baseline,Male,,csf,0.0,Japan
3,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2025,11,counts_baseline_tests_among_eligible,baseline,Male,,pet,0.0,Japan
4,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2025,11,counts_baseline_tests_among_eligible,slow_testing_zero_impact_treatment,Female,,csf,0.0,Japan


In [49]:
csf_pet_tests.query("value>0").age_group.unique()

['40_to_44', '45_to_49', '50_to_54', '55_to_59', '60_to_64', ..., '75_to_79', '80_to_84', '85_to_89', '90_to_94', '95_plus']
Length: 12
Categories (15, object): ['25_to_29' < '30_to_34' < '35_to_39' < '40_to_44' ... '80_to_84' < '85_to_89' < '90_to_94' < '95_plus']

In [50]:
csf_pet_tests_prepped = results.process_csf_pet_tests(csf_pet_tests)
csf_pet_tests_prepped

223.446852 MB measure


31.926852 MB minuend
191.526852 MB subtrahend


21.289441 MB minuend re-indexed
127.689441 MB subtrahend re-indexed


127.689505 MB difference


140.454984 MB difference with reset index
146.839904 MB final difference


,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,testing_state,value,location,metric,disease_stage
0,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2025,11,CSF Tests,baseline,Female,,csf,0.0,Japan,Number,MCI due to AD
1,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2025,11,PET Tests,baseline,Female,,pet,0.0,Japan,Number,MCI due to AD
2,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2025,11,CSF Tests,baseline,Male,,csf,0.0,Japan,Number,MCI due to AD
3,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2025,11,PET Tests,baseline,Male,,pet,0.0,Japan,Number,MCI due to AD
4,30_to_34,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2025,11,CSF Tests,slow_testing_zero_impact_treatment,Female,,csf,0.0,Japan,Number,MCI due to AD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13831995,95_plus,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2100,243,Averted PET Tests,intermediate_testing_with_treatment,Male,,pet,30.0,Israel,Number,MCI due to AD
13831996,95_plus,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2100,243,Averted CSF Tests,accelerated_testing_with_treatment,Female,,csf,7.0,Israel,Number,MCI due to AD
13831997,95_plus,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2100,243,Averted PET Tests,accelerated_testing_with_treatment,Female,,pet,48.0,Israel,Number,MCI due to AD
13831998,95_plus,/mnt/team/simulation_science/pub/models/vivari...,baseline_testing,testing,2100,243,Averted CSF Tests,accelerated_testing_with_treatment,Male,,csf,2.0,Israel,Number,MCI due to AD


In [51]:
# csf_pet_tests_output = results.summarize_and_beautify(csf_pet_tests_prepped, 'disease_stage')
# process_* returns event_year as an unordered categorical, and the
# event_year >= 2025 filter in do_final_processing cannot compare it
# with the installed pandas version, so cast it back to int16 first
csf_pet_tests_output = results.do_final_processing(csf_pet_tests_prepped.astype({'event_year': 'int16'}), include_draws=True)
csf_pet_tests_output = drop_empty_rate_rows(csf_pet_tests_output)
csf_pet_tests_output

2026-11-19 19:42:02.223956


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Timer SummarizingTimer started


Elapsed time: 836.8704 seconds


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:994: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace(column_value_map)


2026-11-19 19:57:14.077273


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243
0,2025,Japan,30_to_34,Female,MCI due to AD,"Slow Testing, No Treatment",Averted CSF Tests,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2025,Japan,30_to_34,Male,MCI due to AD,"Slow Testing, No Treatment",Averted CSF Tests,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2025,Japan,30_to_34,Both,MCI due to AD,"Slow Testing, No Treatment",Averted CSF Tests,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2025,Japan,30_to_34,Female,MCI due to AD,"Intermediate Testing, No Treatment",Averted CSF Tests,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2025,Japan,30_to_34,Male,MCI due to AD,"Intermediate Testing, No Treatment",Averted CSF Tests,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1778395,2100,Israel,65_plus,Male,MCI due to AD,"Intermediate Testing, With Treatment",PET Tests,Rate,0.000494,0.000258,...,0.000696,0.000251,0.000263,0.000325,0.000423,0.000471,0.000399,0.000339,0.000278,0.000797
1778396,2100,Israel,65_plus,Both,MCI due to AD,"Intermediate Testing, With Treatment",PET Tests,Rate,0.000640,0.000323,...,0.000980,0.000325,0.000331,0.000419,0.000537,0.000566,0.000516,0.000458,0.000321,0.001044
1778397,2100,Israel,65_plus,Female,MCI due to AD,"Accelerated Testing, With Treatment",PET Tests,Rate,0.000770,0.000377,...,0.001232,0.000390,0.000390,0.000503,0.000639,0.000651,0.000620,0.000562,0.000358,0.001265
1778398,2100,Israel,65_plus,Male,MCI due to AD,"Accelerated Testing, With Treatment",PET Tests,Rate,0.000494,0.000258,...,0.000696,0.000251,0.000263,0.000325,0.000423,0.000471,0.000399,0.000339,0.000278,0.000797


In [52]:
# Save CSF/PET testing results
save_csv(csf_pet_tests_output, 'csf_pet_tests.csv')

# Treatment Counts

We count treatments using treatment transition counts in older models, and with the treatment duration counts starting in model 11.0

In [53]:
# treatments_filters = [
#     ('sub_entity', 'in',
#      ['susceptible_to_treatment_to_waiting_for_treatment',
#       'waiting_for_treatment_to_full_effect_long',
#       'waiting_for_treatment_to_full_effect_short']
#     )
# ]
treatment_transitions_filename = 'transition_count_treatment'

with Timer():
    treatment_transitions = results.load_measure_from_batch_runs(
        treatment_transitions_filename, False, 
        filters=loading.FINAL_RESULTS_FILTERS[treatment_transitions_filename]
    )
print(len(treatment_transitions), 'rows')
treatment_transitions.head()

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025'), ('sub_entity', 'in', ['waiting_for_treatment_to_full_effect_long', 'waiting_for_treatment_to_full_effect_short', 'waiting_for_treatment_to_treatment_effect']), ('age_group', 'in', ['65_to_69', '70_to_74', '75_to_79', '80_to_84']), ('scenario', 'in', ['slow_testing_with_treatment', 'intermediate_testing_with_treatment', 'accelerated_testing_with_treatment'])]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


1504.812769 MB after read_parquet


9.5828 MB after aggregating seeds and converting dtypes
9.5828 MB total
9.5828 MB after enforcing dtypes
Elapsed time: 1311.8047 seconds
Elapsed time: 0:21:51.804801
456000 rows


,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,value,location
0,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,transition_count,slow_testing_with_treatment,Female,waiting_for_treatment_to_treatment_effect,0.0,Japan
1,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,transition_count,slow_testing_with_treatment,Male,waiting_for_treatment_to_treatment_effect,0.0,Japan
2,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,transition_count,intermediate_testing_with_treatment,Female,waiting_for_treatment_to_treatment_effect,0.0,Japan
3,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,transition_count,intermediate_testing_with_treatment,Male,waiting_for_treatment_to_treatment_effect,0.0,Japan
4,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,transition_count,accelerated_testing_with_treatment,Female,waiting_for_treatment_to_treatment_effect,0.0,Japan


In [54]:
treatment_transitions.sub_entity.unique()

['waiting_for_treatment_to_treatment_effect']
Categories (5, object): ['waiting_for_treatment_to_treatment_effect', 'treatment_effect_to_waning_effect', 'waning_effect_to_no_effect_after_treatment', 'susceptible_to_treatment_to_waiting_for_treat..., 'susceptible_to_treatment_to_no_effect_never_t...]

In [55]:
treatment_transitions.query("value>0").scenario.unique()

['slow_testing_with_treatment', 'intermediate_testing_with_treatment', 'accelerated_testing_with_treatment']
Categories (7, object): ['baseline' < 'slow_testing_zero_impact_treatment' < 'intermediate_testing_zero_impact_treatment' < 'accelerated_testing_zero_impact_treatment' < 'slow_testing_with_treatment' < 'intermediate_testing_with_treatment' < 'accelerated_testing_with_treatment']

In [56]:
treatment_months_filename = 'treatment_duration'
# treatment_months_filters = [
#     ('sub_entity', '!=', 0),
#     ('age_group', 'in', loading.TESTING_ELIGIBLE_AGE_GROUPS),
#     # ('scenario', '==', 'bbbm_testing_and_treatment'),
# ] 

with Timer():
    treatment_months = results.load_measure_from_batch_runs(
        treatment_months_filename, False,
        filters=loading.FINAL_RESULTS_FILTERS[treatment_months_filename]
    )
print(len(treatment_months), 'rows')
treatment_months.head()

Timer BatchLoadingTimer started
[('age_group', '!=', '25_to_29'), ('event_year', '>=', '2025'), ('sub_entity', '!=', 0), ('age_group', 'in', ['65_to_69', '70_to_74', '75_to_79']), ('scenario', 'in', ['slow_testing_with_treatment', 'intermediate_testing_with_treatment', 'accelerated_testing_with_treatment'])]
{'all': PosixPath('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/model16.0/model_spec/2026_11_18_10_15_08/results')}


12312.012104 MB after read_parquet


64.644507 MB after aggregating seeds and converting dtypes
64.644507 MB total
64.644507 MB after enforcing dtypes
Elapsed time: 1412.7096 seconds
Elapsed time: 0:23:32.709759
3078000 rows


,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,value,location
0,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,1,0.0,Japan
1,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,2,0.0,Japan
2,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,3,0.0,Japan
3,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,4,0.0,Japan
4,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,5,0.0,Japan


In [57]:
treatment_months.query("value>0").sub_entity.unique()

[1, 2, 4, 5, 7, 8, 9, 3, 6]
Categories (9, int64): [1, 2, 3, 4, ..., 6, 7, 8, 9]

In [58]:
treatment_months.query("value>0").scenario.unique()

['slow_testing_with_treatment', 'intermediate_testing_with_treatment', 'accelerated_testing_with_treatment']
Categories (7, object): ['baseline' < 'slow_testing_zero_impact_treatment' < 'intermediate_testing_zero_impact_treatment' < 'accelerated_testing_zero_impact_treatment' < 'slow_testing_with_treatment' < 'intermediate_testing_with_treatment' < 'accelerated_testing_with_treatment']

In [59]:
start_treatment = [
        'waiting_for_treatment_to_full_effect_long',
        'waiting_for_treatment_to_full_effect_short',
        'waiting_for_treatment_to_treatment_effect']
treatment_transitions.query("sub_entity in @start_treatment and value>0").age_group.unique()

['65_to_69', '70_to_74', '75_to_79', '80_to_84']
Categories (15, object): ['25_to_29' < '30_to_34' < '35_to_39' < '40_to_44' ... '80_to_84' < '85_to_89' < '90_to_94' < '95_plus']

In [60]:
# # Now empty because we filter out this transition on loading
# treatments.query("sub_entity=='susceptible_to_treatment_to_waiting_for_treatment' and value>0").age_group.unique()
# # Should be ['60_to_64', '65_to_69', '70_to_74', '75_to_79']

In [61]:
# Check which transitions are present in the treatment transitions
# dataframe, to determine whether we should use treatment transition
# counts or treatment duration counts to count completed vs.
# discontinued treatments
unique_transitions = treatment_transitions['sub_entity'].unique()

if ('waiting_for_treatment_to_full_effect_long' in unique_transitions
    and 'waiting_for_treatment_to_full_effect_short' in unique_transitions):
    treatments = treatment_transitions
elif 'waiting_for_treatment_to_treatment_effect' in unique_transitions:
    treatments = treatment_months
else:
    raise ValueError(
        f"Missing expected treatment transitions: {unique_transitions}")

treatments

,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,value,location
0,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,1,0.0,Japan
1,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,2,0.0,Japan
2,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,3,0.0,Japan
3,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,4,0.0,Japan
4,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,5,0.0,Japan
...,...,...,...,...,...,...,...,...,...,...,...,...
3077995,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2100,243,treatment_duration_count,accelerated_testing_with_treatment,Male,5,140.0,Israel
3077996,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2100,243,treatment_duration_count,accelerated_testing_with_treatment,Male,6,136.0,Israel
3077997,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2100,243,treatment_duration_count,accelerated_testing_with_treatment,Male,7,150.0,Israel
3077998,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2100,243,treatment_duration_count,accelerated_testing_with_treatment,Male,8,163.0,Israel


In [62]:
treatments_prepped = results.process_treatments(treatments, mslt_results_prepped)
treatments_prepped

/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:668: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  .query(filter_query)
/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


,age_group,event_year,input_draw,location,scenario,sex,measure,value,disease_stage
0,65_to_69,2025,11,Japan,slow_testing_with_treatment,Female,Medication Discontinuation,0.0,Preclinical AD
1,65_to_69,2025,11,Japan,slow_testing_with_treatment,Female,Medication Completion,0.0,Preclinical AD
2,65_to_69,2025,11,Japan,slow_testing_with_treatment,Male,Medication Discontinuation,0.0,Preclinical AD
3,65_to_69,2025,11,Japan,slow_testing_with_treatment,Male,Medication Completion,0.0,Preclinical AD
4,65_to_69,2025,11,Japan,intermediate_testing_with_treatment,Female,Medication Discontinuation,0.0,Preclinical AD
...,...,...,...,...,...,...,...,...,...
2735995,70_to_74,2100,243,United States of America,accelerated_testing_zero_impact_treatment,Female,Medication Initiation,0.0,Susceptible
2735996,75_to_79,2100,243,United States of America,accelerated_testing_zero_impact_treatment,Female,Medication Initiation,0.0,Susceptible
2735997,65_to_69,2100,243,United States of America,accelerated_testing_zero_impact_treatment,Male,Medication Initiation,0.0,Susceptible
2735998,70_to_74,2100,243,United States of America,accelerated_testing_zero_impact_treatment,Male,Medication Initiation,0.0,Susceptible


In [63]:
treatments_prepped.query("measure=='Medication Initiation'").disease_stage.unique()

['Preclinical AD', 'Susceptible']
Categories (2, object): ['Preclinical AD', 'Susceptible']

In [64]:
treatments_prepped.query("disease_stage=='Susceptible'").scenario.unique()

['slow_testing_with_treatment', 'intermediate_testing_with_treatment', 'accelerated_testing_with_treatment', 'slow_testing_zero_impact_treatment', 'intermediate_testing_zero_impact_treatment', 'accelerated_testing_zero_impact_treatment']
Categories (6, object): ['accelerated_testing_with_treatment', 'accelerated_testing_zero_impact_treatment', 'intermediate_testing_with_treatment', 'intermediate_testing_zero_impact_treatment', 'slow_testing_with_treatment', 'slow_testing_zero_impact_treatment']

In [65]:
treatments_prepped.query("scenario in ['BBBM Testing Only', 'bbbm_testing']").value.unique()

array([], dtype=float64)

In [66]:
with Timer():
    # treatments_output = results.summarize_and_beautify(treatments_prepped, 'disease_stage')
    # process_* returns event_year as an unordered categorical, and the
    # event_year >= 2025 filter in do_final_processing cannot compare it
    # with the installed pandas version, so cast it back to int16 first
    treatments_output = results.do_final_processing(treatments_prepped.astype({'event_year': 'int16'}), include_draws=True)
    treatments_output = drop_empty_rate_rows(treatments_output)

print(len(treatments_output), 'rows')
treatments_output.head()

2026-11-19 20:43:20.235092


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Timer SummarizingTimer started


Elapsed time: 203.4707 seconds
2026-11-19 20:47:03.541026
Elapsed time: 0:03:43.355936
437760 rows


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:994: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace(column_value_map)


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243
0,2025,Brazil,65_plus,Both,Preclinical AD,"Accelerated Testing, With Treatment",Medication Completion,Number,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025,Brazil,65_plus,Female,Preclinical AD,"Accelerated Testing, With Treatment",Medication Completion,Number,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025,Brazil,65_plus,Male,Preclinical AD,"Accelerated Testing, With Treatment",Medication Completion,Number,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025,Brazil,65_plus,Both,Preclinical AD,"Accelerated Testing, No Treatment",Medication Completion,Number,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025,Brazil,65_plus,Female,Preclinical AD,"Accelerated Testing, No Treatment",Medication Completion,Number,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [67]:
treatments_output.query(
    "Year == 2100 & Location == 'United States of America' & Age == '75_to_79' & Sex == 'Male' & `Disease Stage` == 'Susceptible' & Scenario == 'BBBM Testing and Treatment' & Measure == 'Medication Initiation'"
)
# orig mean: 93092.973924
# new mean: 41049.447242

/tmp/ipykernel_872078/2307094547.py:1: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  treatments_output.query(


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243


In [68]:
41049.447242 / 93092.973924

0.4409510783865631

In [69]:
treatments_output.isna().any().any()

False

In [70]:
# Save treatments output
save_csv(treatments_output, 'medication.csv')

# Process months of treatment 

In [71]:
treatment_months
# Needed columns:
# year, location, age group, sex, scenario, sub-entity (months of treatment), value (count)

,age_group,artifact_path,entity,entity_type,event_year,input_draw,measure,scenario,sex,sub_entity,value,location
0,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,1,0.0,Japan
1,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,2,0.0,Japan
2,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,3,0.0,Japan
3,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,4,0.0,Japan
4,65_to_69,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2025,11,treatment_duration_count,slow_testing_with_treatment,Female,5,0.0,Japan
...,...,...,...,...,...,...,...,...,...,...,...,...
3077995,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2100,243,treatment_duration_count,accelerated_testing_with_treatment,Male,5,140.0,Israel
3077996,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2100,243,treatment_duration_count,accelerated_testing_with_treatment,Male,6,136.0,Israel
3077997,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2100,243,treatment_duration_count,accelerated_testing_with_treatment,Male,7,150.0,Israel
3077998,75_to_79,/mnt/team/simulation_science/pub/models/vivari...,treatment,cause,2100,243,treatment_duration_count,accelerated_testing_with_treatment,Male,8,163.0,Israel


In [72]:
treatment_months_prepped = results.process_months_of_treatment(treatment_months)
treatment_months_prepped

/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:710: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  .query(filter_query)


,age_group,event_year,input_draw,measure,scenario,sex,doses_completed,value,location,metric,disease_stage
0,65_to_69,2025,11,Medication by number of doses completed,slow_testing_with_treatment,Female,1,0,Japan,Number,Preclinical AD
1,65_to_69,2025,11,Medication by number of doses completed,slow_testing_with_treatment,Female,2,0,Japan,Number,Preclinical AD
2,65_to_69,2025,11,Medication by number of doses completed,slow_testing_with_treatment,Female,3,0,Japan,Number,Preclinical AD
3,65_to_69,2025,11,Medication by number of doses completed,slow_testing_with_treatment,Female,4,0,Japan,Number,Preclinical AD
4,65_to_69,2025,11,Medication by number of doses completed,slow_testing_with_treatment,Female,5,0,Japan,Number,Preclinical AD
...,...,...,...,...,...,...,...,...,...,...,...
6155995,75_to_79,2100,243,Medication by number of doses completed,accelerated_testing_zero_impact_treatment,Male,5,0,Israel,Number,Preclinical AD
6155996,75_to_79,2100,243,Medication by number of doses completed,accelerated_testing_zero_impact_treatment,Male,6,0,Israel,Number,Preclinical AD
6155997,75_to_79,2100,243,Medication by number of doses completed,accelerated_testing_zero_impact_treatment,Male,7,0,Israel,Number,Preclinical AD
6155998,75_to_79,2100,243,Medication by number of doses completed,accelerated_testing_zero_impact_treatment,Male,8,0,Israel,Number,Preclinical AD


In [73]:
# process_* returns event_year as an unordered categorical, and the
# event_year >= 2025 filter in do_final_processing cannot compare it
# with the installed pandas version, so cast it back to int16 first
treatment_months_output = results.do_final_processing(treatment_months_prepped.astype({'event_year': 'int16'}), include_draws=True)
treatment_months_output = drop_empty_rate_rows(treatment_months_output)
treatment_months_output

2026-11-19 20:47:16.034931


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/vph_output/operations.py:366: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df.replace({category_col: category_to_supercategory})


Timer SummarizingTimer started


Elapsed time: 457.4549 seconds
2026-11-19 20:55:26.372555


/mnt/share/homes/lutzes/vivarium_helpers/src/vivarium_helpers/projects/alzheimers/results.py:994: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  .replace(column_value_map)


,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Doses Completed,Mean,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243
0,2025,Brazil,65_to_69,Female,Preclinical AD,"Accelerated Testing, With Treatment",Medication by number of doses completed,Number,1,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2025,Brazil,65_to_69,Male,Preclinical AD,"Accelerated Testing, With Treatment",Medication by number of doses completed,Number,1,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2025,Brazil,65_to_69,Both,Preclinical AD,"Accelerated Testing, With Treatment",Medication by number of doses completed,Number,1,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2025,Brazil,65_to_69,Female,Preclinical AD,"Accelerated Testing, No Treatment",Medication by number of doses completed,Number,1,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2025,Brazil,65_to_69,Male,Preclinical AD,"Accelerated Testing, No Treatment",Medication by number of doses completed,Number,1,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
991795,2100,United States of America,65_plus,Male,Preclinical AD,"Slow Testing, With Treatment",Medication by number of doses completed,Rate,9,0.001359,...,0.001418,0.001113,0.001391,0.001363,0.001284,0.001408,0.001386,0.001351,0.001413,0.001190
991796,2100,United States of America,65_plus,Both,Preclinical AD,"Slow Testing, With Treatment",Medication by number of doses completed,Rate,9,0.001774,...,0.001887,0.001487,0.001810,0.001741,0.001678,0.001846,0.001722,0.001800,0.001874,0.001585
991797,2100,United States of America,65_plus,Female,Preclinical AD,"Slow Testing, No Treatment",Medication by number of doses completed,Rate,9,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
991798,2100,United States of America,65_plus,Male,Preclinical AD,"Slow Testing, No Treatment",Medication by number of doses completed,Rate,9,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [74]:
treatment_months_output.isna().any().any()

False

In [75]:
# Save months of treatment output
# 41M when including 4 draws, 106M with 25 draws
save_csv(treatment_months_output, 'medication_doses.csv')

# Check some stuff

In [76]:
!ls -halt $output_dir

total 4.8G
-rw-rw-r-- 1 lutzes IHME-Simulationscience  432M Nov 19 20:55 medication_doses.csv
drwxrwsr-x 2 lutzes IHME-Simulationscience  4.0K Nov 19 20:55 .
-rw-rw-r-- 1 lutzes IHME-Simulationscience  183M Nov 19 20:47 medication.csv
-rw-rw-r-- 1 lutzes IHME-Simulationscience  885M Nov 19 19:57 csf_pet_tests.csv
-rw-rw-r-- 1 lutzes IHME-Simulationscience  266M Nov 19 19:27 bbbm_tests.csv
-rw-rw-r-- 1 lutzes IHME-Simulationscience  1.2G Nov 19 18:44 prevalence.csv
-rw-rw-r-- 1 lutzes IHME-Simulationscience  795M Nov 19 18:21 deaths.csv
-rw-rw-r-- 1 lutzes IHME-Simulationscience  136M Nov 19 17:50 population.csv
-rw-rw-r-- 1 lutzes IHME-Simulationscience 1005M Nov 19 15:46 dalys.csv
drwxrwsr-x 3 lutzes IHME-Simulationscience   512 Nov 19 14:06 ..


In [77]:
treatments_output

,Year,Location,Age,Sex,Disease Stage,Scenario,Measure,Metric,Mean,95% UI Lower,...,draw_172,draw_177,draw_179,draw_181,draw_197,draw_198,draw_202,draw_216,draw_233,draw_243
0,2025,Brazil,65_plus,Both,Preclinical AD,"Accelerated Testing, With Treatment",Medication Completion,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2025,Brazil,65_plus,Female,Preclinical AD,"Accelerated Testing, With Treatment",Medication Completion,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2025,Brazil,65_plus,Male,Preclinical AD,"Accelerated Testing, With Treatment",Medication Completion,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2025,Brazil,65_plus,Both,Preclinical AD,"Accelerated Testing, No Treatment",Medication Completion,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2025,Brazil,65_plus,Female,Preclinical AD,"Accelerated Testing, No Treatment",Medication Completion,Number,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
444595,2100,United States of America,75_to_79,Female,Susceptible,"Slow Testing, With Treatment",Medication Initiation,Rate,0.000294,0.000291,...,0.000297,0.000294,0.000295,0.000294,0.000291,0.000294,0.000294,0.000295,0.000295,0.000295
444596,2100,United States of America,75_to_79,Male,Susceptible,"Slow Testing, With Treatment",Medication Initiation,Rate,0.000317,0.000314,...,0.000319,0.000316,0.000317,0.000318,0.000317,0.000319,0.000319,0.000320,0.000318,0.000315
444597,2100,United States of America,75_to_79,Both,Susceptible,"Slow Testing, No Treatment",Medication Initiation,Rate,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
444598,2100,United States of America,75_to_79,Female,Susceptible,"Slow Testing, No Treatment",Medication Initiation,Rate,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [78]:
treatments_output.query("`Disease Stage` == 'Preclinical AD'").Year.unique()

[2025, 2026, 2027, 2028, 2029, ..., 2096, 2097, 2098, 2099, 2100]
Length: 76
Categories (76, int64): [2025, 2026, 2027, 2028, ..., 2097, 2098, 2099, 2100]

In [79]:
treatments_prepped.age_group.unique()

['65_to_69', '70_to_74', '75_to_79']
Categories (3, object): ['65_to_69', '70_to_74', '75_to_79']

In [80]:
treatment_rates = results.calculate_rate(treatments_prepped)
treatment_rates

,age_group,disease_stage,event_year,input_draw,location,measure,scenario,sex,value,metric
0,65_to_69,NaN,2025,11,Brazil,NaN,accelerated_testing_with_treatment,Both,NaN,Rate
1,65_to_69,Preclinical AD,2025,11,Brazil,Medication Discontinuation,accelerated_testing_with_treatment,Female,0.0,Rate
2,65_to_69,Preclinical AD,2025,11,Brazil,Medication Completion,accelerated_testing_with_treatment,Female,0.0,Rate
3,65_to_69,Preclinical AD,2025,11,Brazil,Medication Initiation,accelerated_testing_with_treatment,Female,0.0,Rate
4,65_to_69,Susceptible,2025,11,Brazil,Medication Initiation,accelerated_testing_with_treatment,Female,0.0,Rate
...,...,...,...,...,...,...,...,...,...,...
3248995,75_to_79,Susceptible,2100,243,United States of America,Medication Initiation,slow_testing_zero_impact_treatment,Female,0.0,Rate
3248996,75_to_79,Preclinical AD,2100,243,United States of America,Medication Discontinuation,slow_testing_zero_impact_treatment,Male,0.0,Rate
3248997,75_to_79,Preclinical AD,2100,243,United States of America,Medication Completion,slow_testing_zero_impact_treatment,Male,0.0,Rate
3248998,75_to_79,Preclinical AD,2100,243,United States of America,Medication Initiation,slow_testing_zero_impact_treatment,Male,0.0,Rate


In [81]:
treatment_rates.query("disease_stage=='Susceptible'")

,age_group,disease_stage,event_year,input_draw,location,measure,scenario,sex,value,metric
4,65_to_69,Susceptible,2025,11,Brazil,Medication Initiation,accelerated_testing_with_treatment,Female,0.000000,Rate
8,65_to_69,Susceptible,2025,11,Brazil,Medication Initiation,accelerated_testing_with_treatment,Male,0.000000,Rate
13,65_to_69,Susceptible,2025,11,Brazil,Medication Initiation,accelerated_testing_zero_impact_treatment,Female,0.000000,Rate
17,65_to_69,Susceptible,2025,11,Brazil,Medication Initiation,accelerated_testing_zero_impact_treatment,Male,0.000000,Rate
25,65_to_69,Susceptible,2025,11,Brazil,Medication Initiation,intermediate_testing_with_treatment,Female,0.000000,Rate
...,...,...,...,...,...,...,...,...,...,...
3248981,75_to_79,Susceptible,2100,243,United States of America,Medication Initiation,intermediate_testing_zero_impact_treatment,Male,0.000000,Rate
3248986,75_to_79,Susceptible,2100,243,United States of America,Medication Initiation,slow_testing_with_treatment,Female,0.000083,Rate
3248990,75_to_79,Susceptible,2100,243,United States of America,Medication Initiation,slow_testing_with_treatment,Male,0.000089,Rate
3248995,75_to_79,Susceptible,2100,243,United States of America,Medication Initiation,slow_testing_zero_impact_treatment,Female,0.000000,Rate


In [82]:
mslt_results_prepped.query("measure.str.contains('Medication', na=False)", engine="python")

,age_group,disease_stage,event_year,input_draw,location,measure,metric,scenario,sex,value
912,65_to_69,Susceptible,2025,11,Brazil,Medication Initiation,Number,slow_testing_zero_impact_treatment,Female,0.000000
913,70_to_74,Susceptible,2025,11,Brazil,Medication Initiation,Number,slow_testing_zero_impact_treatment,Female,0.000000
914,75_to_79,Susceptible,2025,11,Brazil,Medication Initiation,Number,slow_testing_zero_impact_treatment,Female,0.000000
915,65_to_69,Susceptible,2025,11,Brazil,Medication Initiation,Number,slow_testing_zero_impact_treatment,Male,0.000000
916,70_to_74,Susceptible,2025,11,Brazil,Medication Initiation,Number,slow_testing_zero_impact_treatment,Male,0.000000
...,...,...,...,...,...,...,...,...,...,...
2051995,70_to_74,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Female,678.402772
2051996,75_to_79,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Female,656.663384
2051997,65_to_69,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Male,821.539394
2051998,70_to_74,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Male,619.059922


In [83]:
mslt_results_prepped.measure.unique()

['BBBM Tests', 'Positive BBBM Tests', 'Medication Initiation']
Categories (3, object): ['BBBM Tests', 'Medication Initiation', 'Positive BBBM Tests']

# Scratchwork

In [84]:
bbbm_tests_prepped

,age_group,event_year,input_draw,location,measure,scenario,sex,value,disease_stage,metric
0,65_to_69,2025,11,Japan,BBBM Tests,slow_testing_zero_impact_treatment,Female,0.000000,Preclinical AD,Number
1,65_to_69,2025,11,Japan,BBBM Tests,slow_testing_zero_impact_treatment,Male,0.000000,Preclinical AD,Number
2,65_to_69,2025,11,Japan,BBBM Tests,intermediate_testing_zero_impact_treatment,Female,0.000000,Preclinical AD,Number
3,65_to_69,2025,11,Japan,BBBM Tests,intermediate_testing_zero_impact_treatment,Male,0.000000,Preclinical AD,Number
4,65_to_69,2025,11,Japan,BBBM Tests,accelerated_testing_zero_impact_treatment,Female,0.000000,Preclinical AD,Number
...,...,...,...,...,...,...,...,...,...,...
2735995,70_to_74,2100,243,United States of America,Positive BBBM Tests,accelerated_testing_with_treatment,Female,848.003465,Susceptible,Number
2735996,75_to_79,2100,243,United States of America,Positive BBBM Tests,accelerated_testing_with_treatment,Female,820.829230,Susceptible,Number
2735997,65_to_69,2100,243,United States of America,Positive BBBM Tests,accelerated_testing_with_treatment,Male,1026.924243,Susceptible,Number
2735998,70_to_74,2100,243,United States of America,Positive BBBM Tests,accelerated_testing_with_treatment,Male,773.824903,Susceptible,Number


In [85]:
column_name_map = {
            'event_year': 'Year',
            'age_group': 'Age',
            'location': 'Location',
            'sex': 'Sex',
            'scenario': 'Scenario',
            'measure': 'Measure',
            'metric': 'Metric',
            'disease_stage': 'Disease Stage',
            'input_draw': 'Draw',
            'value': 'Value',
            'mean': 'Mean',
            'lower': '95% UI Lower',
            'upper': '95% UI Upper',
        }
reverse_map = {v: k for k, v in column_name_map.items()}
reverse_map

{'Year': 'event_year',
 'Age': 'age_group',
 'Location': 'location',
 'Sex': 'sex',
 'Scenario': 'scenario',
 'Measure': 'measure',
 'Metric': 'metric',
 'Disease Stage': 'disease_stage',
 'Draw': 'input_draw',
 'Value': 'value',
 'Mean': 'mean',
 '95% UI Lower': 'lower',
 '95% UI Upper': 'upper'}

In [86]:
temp = mslt_results.rename(columns=reverse_map)
temp

,event_year,location,age_group,sex,disease_stage,scenario,measure,metric,input_draw,value
0,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_105,0.000000
1,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_11,0.000000
2,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_135,0.000000
3,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_15,0.000000
4,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_156,0.000000
...,...,...,...,...,...,...,...,...,...,...
113995,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Improper Medication Uses,Number,draw_35,2164.945160
113996,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Improper Medication Uses,Number,draw_41,2092.434909
113997,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Improper Medication Uses,Number,draw_75,2051.974214
113998,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Improper Medication Uses,Number,draw_90,2003.006205


In [87]:
pd.concat([bbbm_tests_prepped, temp])

,age_group,event_year,input_draw,location,measure,scenario,sex,value,disease_stage,metric
0,65_to_69,2025,11,Japan,BBBM Tests,slow_testing_zero_impact_treatment,Female,0.000000,Preclinical AD,Number
1,65_to_69,2025,11,Japan,BBBM Tests,slow_testing_zero_impact_treatment,Male,0.000000,Preclinical AD,Number
2,65_to_69,2025,11,Japan,BBBM Tests,intermediate_testing_zero_impact_treatment,Female,0.000000,Preclinical AD,Number
3,65_to_69,2025,11,Japan,BBBM Tests,intermediate_testing_zero_impact_treatment,Male,0.000000,Preclinical AD,Number
4,65_to_69,2025,11,Japan,BBBM Tests,accelerated_testing_zero_impact_treatment,Female,0.000000,Preclinical AD,Number
...,...,...,...,...,...,...,...,...,...,...
113995,75_to_79,2100,draw_35,United States of America,Improper Medication Uses,"Accelerated Testing, With Treatment",Male,2164.945160,Susceptible,Number
113996,75_to_79,2100,draw_41,United States of America,Improper Medication Uses,"Accelerated Testing, With Treatment",Male,2092.434909,Susceptible,Number
113997,75_to_79,2100,draw_75,United States of America,Improper Medication Uses,"Accelerated Testing, With Treatment",Male,2051.974214,Susceptible,Number
113998,75_to_79,2100,draw_90,United States of America,Improper Medication Uses,"Accelerated Testing, With Treatment",Male,2003.006205,Susceptible,Number


In [88]:
temp.query("measure.str.contains('Medication', na=False) and value > 0 and scenario=='BBBM Testing Only'", engine="python")

,event_year,location,age_group,sex,disease_stage,scenario,measure,metric,input_draw,value


In [89]:
temp.measure.unique()

array(['BBBM Tests', 'BBBM False Positive Tests',
       'Improper Medication Uses'], dtype=object)

In [90]:
temp.replace(
    {'measure': {'BBBM False Positive Tests': 'BBBM Positive Tests',
 'Improper Medication Uses': 'Medication Initiation'}})


,event_year,location,age_group,sex,disease_stage,scenario,measure,metric,input_draw,value
0,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_105,0.000000
1,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_11,0.000000
2,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_135,0.000000
3,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_15,0.000000
4,2025,Brazil,65_to_69,Female,Susceptible,"Slow Testing, No Treatment",BBBM Tests,Number,draw_156,0.000000
...,...,...,...,...,...,...,...,...,...,...
113995,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Medication Initiation,Number,draw_35,2164.945160
113996,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Medication Initiation,Number,draw_41,2092.434909
113997,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Medication Initiation,Number,draw_75,2051.974214
113998,2100,United States of America,75_to_79,Male,Susceptible,"Accelerated Testing, With Treatment",Medication Initiation,Number,draw_90,2003.006205


In [91]:
bbbm_tests_prepped.dtypes

age_group        category
event_year       category
input_draw       category
location         category
measure          category
scenario         category
sex              category
value             float64
disease_stage    category
metric           category
dtype: object

In [92]:
testing = temp.query("scenario=='BBBM Testing Only'").drop(columns='scenario')
treatment = temp.query("scenario=='BBBM Testing and Treatment'").drop(columns='scenario')
# This shows that the whole dataframe was copied to the testing-only
# scenario, including medication initiations, which is incorrect
results.ops.compare_values(testing, treatment)

age_group,disease_stage,event_year,input_draw,location,measure,metric,sex


In [93]:
mslt_results_processed = results.process_mslt_results(mslt_results)
mslt_results_processed

,age_group,disease_stage,event_year,input_draw,location,measure,metric,scenario,sex,value
0,65_to_69,Susceptible,2025,11,Brazil,BBBM Tests,Number,slow_testing_zero_impact_treatment,Female,0.000000
1,70_to_74,Susceptible,2025,11,Brazil,BBBM Tests,Number,slow_testing_zero_impact_treatment,Female,0.000000
2,75_to_79,Susceptible,2025,11,Brazil,BBBM Tests,Number,slow_testing_zero_impact_treatment,Female,0.000000
3,65_to_69,Susceptible,2025,11,Brazil,BBBM Tests,Number,slow_testing_zero_impact_treatment,Male,0.000000
4,70_to_74,Susceptible,2025,11,Brazil,BBBM Tests,Number,slow_testing_zero_impact_treatment,Male,0.000000
...,...,...,...,...,...,...,...,...,...,...
2051995,70_to_74,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Female,678.402772
2051996,75_to_79,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Female,656.663384
2051997,65_to_69,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Male,821.539394
2051998,70_to_74,Susceptible,2100,243,United States of America,Medication Initiation,Number,accelerated_testing_with_treatment,Male,619.059922


In [94]:
mslt_results_processed.input_draw.unique()

[11, 15, 19, 24, 26, ..., 198, 202, 216, 233, 243]
Length: 25
Categories (25, int16): [11, 15, 19, 24, ..., 202, 216, 233, 243]

In [95]:
print_memory_usage(mslt_results_processed)

34.89015 MB 


In [96]:
mslt_results_processed.measure.unique()

['BBBM Tests', 'Positive BBBM Tests', 'Medication Initiation']
Categories (3, object): ['BBBM Tests', 'Medication Initiation', 'Positive BBBM Tests']

In [97]:
!date

Thu Nov 19 20:56:03 PST 2026
